In [ ]:
import json
import re
from collections import defaultdict

CONFIG_FOLDER = '../src/config/'
DATA_FOLDER = '../data/'

config_file_path = CONFIG_FOLDER + 'esg_metric_dictionary.json'

# Load the JSON data into a dictionary
with open(config_file_path, 'r') as file:
    data_dict = json.load(file)

# Print the loaded dictionary
# print(data_dict)

# Accessing specific values
# print("Carbon Emissions Description:", data_dict["carbon_emissions"])
# print("Renewable Energy Description:", data_dict["renewable_energy"])


# Load the text from the file
file_path = DATA_FOLDER + 'citybank2019_text_extracted.txt'

with open(file_path, 'r') as file:
    loaded_text = file.read()

# Print the loaded text
print(loaded_text)


# Step 1: Extract key-value pairs from the text
kv_pattern = r'(\w+)\s+([^ ]+)'
kv_matches = re.findall(kv_pattern, loaded_text)

# Step 2: Define label mapping for keys
label_mapping = data_dict
# label_mapping = {
#     'key1': 'L1',
#     'key2': 'L2',
#     'key3': 'L3'
# }

# Step 3: Create a defaultdict to collect values by label
output_dict = defaultdict(list)

# Step 4: Populate the output dictionary
for key, value in kv_matches:
    if key in label_mapping:
        label = label_mapping[key]
        output_dict[label].append(value)

# Convert defaultdict to a regular dictionary for output
final_output = dict(output_dict)

# Print the final output
print(final_output)

In [1]:
import re
from collections import defaultdict

# Input text
text = "bbb key1 value1 abcs key2 value2 xyz key3 value3 aaa key1 value4 bbb"

# Step 1: Extract key-value pairs from the text
kv_pattern = r'(\w+)\s+([^ ]+)'
kv_matches = re.findall(kv_pattern, text)

# Step 2: Define the label mapping for keys
label_mapping = {
    'key1': 'L1',
    'key2': 'L2',
    'key3': 'L3'
}

# Step 3: Create a defaultdict to collect values by label
output_dict = defaultdict(list)

# Step 4: Populate the output dictionary
for key, value in kv_matches:
    # Check if the extracted key exists in the label mapping
    if key in label_mapping:
        label = label_mapping[key]
        output_dict[label].append(value)

# Convert defaultdict to a regular dictionary for output
final_output = dict(output_dict)

# Print the final output
print(final_output)

{'L2': ['value2'], 'L1': ['value4']}


In [3]:
import re

# Sample ESG text
esg_text = """Our company emitted 2000 tons of CO2 in 2023. The diversity ratio is 45%. 
Renewable energy usage has increased to 60%. We have 30% women in leadership roles."""

labels_metrics = {
    'carbon_emissions': r'(\d+)\s*tons?\s*of\s*CO2',
    'diversity_ratio': r'(\d+)%\s*diversity\s*ratio',
    'renewable_energy_usage': r'(\d+)%\s*renewable\s*energy\s*usage',
    'women_in_leadership': r'(\d+)%\s*women\s*in\s*leadership\s*roles'
}

# Dictionary to hold the extracted data
extracted_data = {}

# Extract data based on defined labels and metrics
for label, pattern in labels_metrics.items():
    match = re.search(pattern, esg_text)
    if match:
        extracted_data[label] = match.group(1)  # Get the first capturing group

# Print the extracted data
print(extracted_data)

{'carbon_emissions': '2000', 'women_in_leadership': '30'}


In [18]:
import spacy
import nltk
import re

# Load spaCy model
nlp = spacy.load('en_core_web_sm')

# Sample ESG text
esg_text = """Our company emitted 2000 tons of CO2 in 2023. We aim for a diversity ratio of 45%. 
Renewable energy usage has risen to 60% this year. Currently, women make up 30% of leadership roles."""

esg_text_list = ['Our company emitted 2000 tons of CO2 in 2023.', 'We aim for a diversity ratio of 45%.', 'Renewable energy usage has risen to 60% this year.', 'Currently, women make up 30% of leadership roles.']

# Process text with spaCy
doc = nlp(esg_text)

# Define a dictionary to store extracted data
extracted_data = {}

print('doc', doc)
# print('doc.ents', doc.ents)

# Extract entity data using spaCy
for ent in doc.ents:
    if ent.label_ == "QUANTITY":  # Recognize numerical quantities
        extracted_data['emission'] = extracted_data.get('emission', []) + [ent.text]

# Print the results of NER
print("Quantitative Data from spaCy NER:", extracted_data)

# Define patterns for quantitative data extraction using regex
patterns = {
    'carbon_emissions': r'(\d+)\s*tons?\s*of\s*CO2',
    'diversity_ratio': r'(\d+)%\s*diversity\s*ratio',
    'renewable_energy_usage': r'(\d+)%\s*renewable\s*energy\s*usage',
    'women_in_leadership': r'(\d+)%\s*women\s*make\s*up\s*leadership\s*roles'
}

# Extract data based on defined patterns with regex
for label, pattern in patterns.items():
    match = re.search(pattern, esg_text)
    if match:
        extracted_data[label] = match.group(1)

# Print the combined results
print("Quantitative Data from regex patterns:", extracted_data)


# Combine results from both extraction methods
final_extracted_data = {
    'carbon_emissions': extracted_data.get('carbon_emissions', []),
    'diversity_ratio': extracted_data.get('diversity_ratio', []),
    'renewable_energy_usage': extracted_data.get('renewable_energy_usage', []),
    'women_in_leadership': extracted_data.get('women_in_leadership', [])
}

# Print final output
print("Final Extracted Data:", final_extracted_data)

doc Our company emitted 2000 tons of CO2 in 2023. We aim for a diversity ratio of 45%. 
Renewable energy usage has risen to 60% this year. Currently, women make up 30% of leadership roles.
Quantitative Data from spaCy NER: {'emission': ['2000 tons']}
Quantitative Data from regex patterns: {'emission': ['2000 tons'], 'carbon_emissions': '2000'}
Final Extracted Data: {'carbon_emissions': '2000', 'diversity_ratio': [], 'renewable_energy_usage': [], 'women_in_leadership': []}


In [26]:
# Gensim
import gensim
import gensim.corpora as corpora
from gensim.utils import simple_preprocess
from gensim.models import CoherenceModel

# Plotting tools
import pyLDAvis
import pyLDAvis.gensim  # don't skip this

# NLTK Stop words
from nltk.corpus import stopwords
stop_words = stopwords.words('english')

from sklearn.feature_extraction import text
stop_words = text.ENGLISH_STOP_WORDS.union(stop_words)

# context specific keywords not to include in topic modelling
fsi_stop_words = [
  'plc', 'group', 'target',
  'track', 'capital', 'holding',
  'report', 'annualreport',
  'esg', 'bank', 'report',
  'annualreport', 'long', 'make', 'diversity ratio', 'co2'
]

# fsi_stop_words.append(report_company)
fsi_stop_words.append('citi')

# our list contains all english stop words + companies names + specific keywords
stop_words = stop_words.union(fsi_stop_words)


def run_NLP(content):

    def sent_to_words(sentences):
        for sentence in sentences:
            yield(gensim.utils.simple_preprocess(str(sentence), deacc=True))  # deacc=True removes punctuations

    # Define functions for stopwords, bigrams, trigrams and lemmatization
    def remove_stopwords(texts):
        return [[word for word in simple_preprocess(str(doc)) if word not in stop_words] for doc in texts]

    def make_bigrams(texts):
        return [bigram_mod[doc] for doc in texts]

    def make_trigrams(texts):
        return [trigram_mod[bigram_mod[doc]] for doc in texts]

    def lemmatization(texts, allowed_postags=['NOUN', 'ADJ', 'VERB', 'ADV']):
        """https://spacy.io/api/annotation"""
        texts_out = []
        for sent in texts:
            doc = nlp(" ".join(sent)) 
            texts_out.append([token.lemma_ for token in doc if token.pos_ in allowed_postags])
        return texts_out

    print('content:', content)
    data_words = list(sent_to_words(content))
    print('data_words:', data_words)

    # Build the bigram and trigram models
    bigram = gensim.models.Phrases(data_words, min_count=5, threshold=100) # higher threshold fewer phrases.
    print('bigram:', bigram)
    trigram = gensim.models.Phrases(bigram[data_words], threshold=100)
    print('trigram:', trigram)

    # Faster way to get a sentence clubbed as a trigram/bigram
    bigram_mod = gensim.models.phrases.Phraser(bigram)
    print('bigram_mod:', bigram_mod)
    trigram_mod = gensim.models.phrases.Phraser(trigram)
    print('trigram_mod:', trigram_mod)

    # Remove Stop Words
    data_words_nostops = remove_stopwords(data_words)
    print('data_words_nostops:', data_words_nostops)

    # Form Bigrams
    data_words_bigrams = make_bigrams(data_words_nostops)
    print('data_words_bigrams:', data_words_bigrams)

    # Do lemmatization keeping only noun, adj, vb, adv
    data_lemmatized = lemmatization(data_words_bigrams, allowed_postags=['NOUN', 'ADJ', 'VERB', 'ADV'])
    print('data_lemmatized:', data_lemmatized)
    
    return data_lemmatized

In [27]:
# print('stop_words: ' , ', '.join(stop_words))
print(esg_text_list)
data_lemmatized = run_NLP(esg_text_list)
report_sentences_lemma = [' '.join(w) for w in data_lemmatized]
# import random
# report_sentences_lemma[random.randint(0, len(report_sentences_lemma))]
# print(data_lemmatized)

['Our company emitted 2000 tons of CO2 in 2023.', 'We aim for a diversity ratio of 45%.', 'Renewable energy usage has risen to 60% this year.', 'Currently, women make up 30% of leadership roles.']
content: ['Our company emitted 2000 tons of CO2 in 2023.', 'We aim for a diversity ratio of 45%.', 'Renewable energy usage has risen to 60% this year.', 'Currently, women make up 30% of leadership roles.']
data_words: [['our', 'company', 'emitted', 'tons', 'of', 'co', 'in'], ['we', 'aim', 'for', 'diversity', 'ratio', 'of'], ['renewable', 'energy', 'usage', 'has', 'risen', 'to', 'this', 'year'], ['currently', 'women', 'make', 'up', 'of', 'leadership', 'roles']]
bigram: Phrases<50 vocab, min_count=5, threshold=100, max_vocab_size=40000000>
trigram: Phrases<50 vocab, min_count=5, threshold=100, max_vocab_size=40000000>
bigram_mod: FrozenPhrases<0 phrases, min_count=5, threshold=100>
trigram_mod: FrozenPhrases<0 phrases, min_count=5, threshold=100>
data_words_nostops: [['company', 'emitted', 'ton